# Stage 2: QC + 过滤 + 双细胞鉴定（scrublet）+ 环境 RNA 校正（SoupX，条件性）

按来源数据集逐个进行质量控制，通过**manifest 驱动跳过逻辑**：

- **双细胞鉴定（doublet detection）**：当来源数据集已被原作者去除双细胞时跳过
  （在 `manifest.yaml` 的 `preprocessing_done` 或 `qc_overrides` 中声明）。
- **环境 RNA 校正（SoupX）**：仅当 manifest 声明了 `input.raw_path`
  **且**运行时 R 环境可用时才尝试。R 不可用时优雅跳过，不崩溃。
- **基础过滤**：`min_genes`、`max_genes`、`max_pct_mt` 阈值。

**为什么要按来源分步做？** 不同数据集前处理历史不同——
有的原作者已去过双细胞，重跑是科学错误。manifest 记录了每套数据的真实状态，
notebook 据此决定跑还是跳过。

**本 notebook 产出**：
- `obs.doublet_score` / `predicted_doublet`——跨所有细胞对齐（跳过时写 NaN/False）
- `obs.ambient_correction_applied`——逐细胞布尔值
- `adata.uns['qc_skipped']` / `"qc_heterogeneous"` / `"filter_v1"`——结构化 QC 记录
- 过滤前后 QC 可视化（供 before/after 对比）
- Stage 2 checkpoint `.h5ad` 文件，供 stage 3 标准化使用

In [ ]:
# === PARAMS ===
# UPSTREAM_PATH -- stage 1 output from a prior notebook run.
# OUTPUT_PATH   -- where to write this stage's checkpoint.
# The manifest path is derived from metadata in the h5ad;
# QC thresholds are PI-tunable knobs (per ADR-0006, knobs live here, not in manifest).

UPSTREAM_PATH = "results/nancang_stage1_loaded_v1.h5ad"
OUTPUT_PATH   = "results/nancang_stage2_qcd_v1.h5ad"

# QC filter thresholds (tune per project and dataset)
MIN_GENES     = 200
MAX_GENES     = 6000
MAX_PCT_MT    = 20
RANDOM_SEED   = 42


In [ ]:
# Ensure the framework src/ is on sys.path and CWD is set to the project root.
# Detects: if running from notebooks/ (Jupyter) or from project root (nbconvert).
import sys, os
_root = os.getcwd()
if not os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
    _root = os.path.abspath(os.path.join(_root, ".."))
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")


In [ ]:
# Imports.
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
import warnings
from pathlib import Path

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)

import importlib.metadata; print(f"Scanpy {importlib.metadata.version('scanpy')}  |  anndata {importlib.metadata.version('anndata')}")


In [ ]:
# Load the stage 1 h5ad.
print(f"Loading upstream: {UPSTREAM_PATH}")
adata = sc.read_h5ad(UPSTREAM_PATH)
print(f"Loaded: {adata.n_obs:,} cells x {adata.n_vars:,} genes")

# Identify source datasets present (stage 2 must handle multi-source).
source_datasets = sorted(adata.obs['source_dataset'].unique())
print(f"Source datasets in this AnnData: {source_datasets}")

In [ ]:
# Load per-source manifests for preprocessing_done / qc_overrides.
# Stage 2 reads the manifest to decide which QC steps to skip
# (e.g. author already removed doublets — re-running would be a scientific error).
manifests = {}
MANIFEST_BASE = "data"  # manifests live at data/{source_dataset}/manifest.yaml

for src in source_datasets:
    mf_path = Path(MANIFEST_BASE) / src.lower().replace("_", "") / "manifest.yaml"
    # Try common name patterns
    candidates = [
        Path(MANIFEST_BASE) / src.lower() / "manifest.yaml",
        Path(MANIFEST_BASE) / src.lower().split("_")[0] / "manifest.yaml",
    ]
    found = None
    for c in candidates:
        if c.exists():
            found = c
            break
    # Also try direct lookup in data/ subdirs
    if found is None:
        for sub in Path(MANIFEST_BASE).iterdir():
            if sub.is_dir() and (sub / "manifest.yaml").exists():
                mf = yaml.safe_load((sub / "manifest.yaml").read_text())
                if mf.get("source_dataset") == src:
                    found = sub / "manifest.yaml"
                    break
    if found:
        manifests[src] = yaml.safe_load(found.read_text())
        print(f"  {src}: manifest loaded from {found}")
    else:
        print(f"  {src}: manifest NOT found — assuming no QC skips")
        manifests[src] = {}

# Summarize preprocessing state across sources.
print("\n===== Preprocessing state per source =====")
for src in source_datasets:
    mf = manifests.get(src, {})
    pp = mf.get("preprocessing_done", [])
    qc_overrides = mf.get("qc_overrides", {})
    print(f"  {src}: preprocessing_done={pp}")
    if qc_overrides:
        print(f"         qc_overrides={list(qc_overrides.keys())}")

## QC visualisation pre-filter

Per-source violin and scatter plots of the three main QC metrics
(`n_genes`, `total_counts`, `pct_counts_mt`). These are computed on raw counts
and give PI a first look at data quality before any filtering.

In [ ]:
# QC visualisation: violin plots per source_dataset.
# PI inspects these to set filter thresholds for the project.
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, metric in enumerate(["n_genes", "total_counts", "pct_counts_mt"]):
    ax = axes[i]
    sc.pl.violin(adata, keys=metric, groupby="source_dataset",
                rotation=45, ax=ax, show=False)
    ax.set_title(f"{metric} (pre-filter)")
plt.tight_layout()
fig.savefig("results/figures/stage2_qc_violin_pre.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# QC visualisation: scatter plots (n_genes vs pct_mt, total_counts vs n_genes).
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sc.pl.scatter(adata, x="total_counts", y="n_genes", color="pct_counts_mt",
             ax=axes[0], show=False)
axes[0].set_title("total_counts vs n_genes, colored by pct_mt (pre-filter)")
sc.pl.scatter(adata, x="n_genes", y="pct_counts_mt", color="total_counts",
             ax=axes[1], show=False)
axes[1].set_title("n_genes vs pct_mt, colored by total_counts (pre-filter)")
plt.tight_layout()
fig.savefig("results/figures/stage2_qc_scatter_pre.png", dpi=150, bbox_inches="tight")
plt.show()

## Doublet detection (manifest-driven skip logic)

Per-source-dataset doublet calling:
- **Skip** when `preprocessing_done` includes `"doublet_removal"`
  or `qc_overrides.doublet_removal.skip` is true.
  The `qc_overrides.doublet_removal.reason` is mandatory when skip is true (SPEC manifest schema).
  Skipped cells get `obs.doublet_score = NaN` and `obs.predicted_doublet = False`
  (NaN encodes "not applicable", not "missing data" — SPEC QC Heterogeneity).
- **Run scrublet** on the per-source subset when the source is not skipped.

In [ ]:
# 双细胞鉴定——按来源数据集，manifest 驱动跳过或运行。
# 每套数据可能有不同的双细胞处理历史；notebook 透明处理这种异构性。
# 列对齐：每套数据都获得相同的 obs 列，跳过者用 NaN 表示"不适用"。

# 初始化双细胞列（如不存在）。
if "doublet_score" not in adata.obs.columns:
    adata.obs["doublet_score"] = np.nan
    adata.obs["predicted_doublet"] = False

qc_doublet_skipped = {}  # 记录每来源数据集的跳过情况，供 uns["qc_skipped"] 用

for src in source_datasets:
    src_mask = adata.obs["source_dataset"] == src
    n_src = src_mask.sum()
    mf = manifests.get(src, {})
    pp_done = mf.get("preprocessing_done", [])
    qc_overrides = mf.get("qc_overrides", {})
    doublet_override = qc_overrides.get("doublet_removal", {})

    skip = False
    skip_reason = None

    if "doublet_removal" in pp_done:
        skip = True
        skip_reason = "原作者已去除双细胞（preprocessing_done 含 doublet_removal）"
    elif doublet_override.get("skip"):
        skip = True
        skip_reason = doublet_override.get("reason", "qc_overrides.doublet_removal.skip=True（无理由）")

    if skip:
        print(f"  {src} ({n_src} 细胞): 跳过 doublet_removal → {skip_reason}")
        adata.obs.loc[src_mask, "doublet_score"] = np.nan
        adata.obs.loc[src_mask, "predicted_doublet"] = False
        qc_doublet_skipped[src] = {"step": "doublet_removal", "reason": skip_reason}
    else:
        print(f"  {src} ({n_src} 细胞): 运行 scrublet...")
        sub_adata = adata[src_mask].copy()
        sc.external.pp.scrublet(sub_adata, random_state=RANDOM_SEED)
        adata.obs.loc[src_mask, "doublet_score"] = sub_adata.obs["doublet_score"].values
        adata.obs.loc[src_mask, "predicted_doublet"] = sub_adata.obs["predicted_doublet"].values
        n_doublets = int(sub_adata.obs["predicted_doublet"].sum())
        print(f"    → {n_doublets} 个预测双细胞 ({100*n_doublets/n_src:.1f}%)")
        qc_doublet_skipped[src] = {"step": "doublet_removal", "ran": True, "n_doublets": n_doublets}

adata.obs["predicted_doublet"] = adata.obs["predicted_doublet"].astype(bool)
print("\n各来源双细胞数:", adata.obs.groupby("source_dataset")["predicted_doublet"].sum().to_dict())


## Ambient RNA correction via SoupX (conditional)

SoupX corrects for ambient RNA contamination using the raw (unfiltered) count matrix.
It is **conditional on three factors**:

1. **Manifest has `input.raw_path`** — the raw_feature_bc_matrix directory.
   Some datasets only ship filtered matrices; SoupX is physically impossible.
2. **R environment with SoupX is available** — checked at runtime via `rpy2`.
   If R is not installed or `SoupX` R package is missing, the cell prints a clear
   skip message and sets `obs.ambient_correction_applied = False`.
   **The notebook does NOT crash when R is unavailable.**
3. **Per-sample execution** — SoupX runs on one sample at a time (per the legacy-GCPL
   pattern), passing filtered counts, raw counts, and cluster labels across the
   Python-R boundary via `rpy2`.

In [ ]:
# Ambient correction — conditional on raw_path AND R/SoupX availability.
# Guard 1: check for raw matrix path.
raw_path = adata.uns.get("raw_matrix_path", None)

# Initialize ambient correction column.
if "ambient_correction_applied" not in adata.obs.columns:
    adata.obs['ambient_correction_applied'] = False

soupx_ran = False

if raw_path is None:
    print("SoupX SKIPPED: no raw_matrix_path in adata.uns (manifest input.raw_path not declared).")
    print("  Ambient correction is physically impossible for datasets that ship only filtered matrices.")
else:
    print(f"raw_matrix_path found: {raw_path}")

    # Guard 2: check R / rpy2 / SoupX availability.
    r_available = False
    try:
        import rpy2
        from rpy2.robjects.packages import importr
        import anndata2ri
        SoupX_pkg = importr('SoupX')
        r_available = True
        print("  R + rpy2 + anndata2ri + SoupX: available")
    except Exception as e:
        print(f"  R / rpy2 / SoupX NOT available: {e}")
        print("  SoupX SKIPPED: R environment not available. Install with:")
        print("    conda env create -f environment-r.yml")
        print("    conda activate scrna-integration-r")
        print('  or install SoupX in R: install.packages("SoupX")')

    if r_available:
        # Guard 3: per-sample SoupX execution (legacy-GCPL pattern).
        # The raw_path for Nancang points to the parent dir containing
        # subdirs with raw_feature_bc_matrix; iterate over samples.
        raw_dir = Path(raw_path)
        import anndata

        for src in source_datasets:
            src_mask = adata.obs['source_dataset'] == src
            samples = sorted(adata.obs.loc[src_mask, "sample_id"].unique())
            print(f"\n  {src}: {len(samples)} samples")

            for sample_id in samples:
                sample_mask = src_mask & (adata.obs['sample_id'] == sample_id)
                sample_barcodes = adata.obs_names[sample_mask]
                print(f"    {sample_id}: {sample_mask.sum()} cells")

                # Locate the raw_feature_bc_matrix for this sample.
                raw_sample_dir = raw_dir / sample_id / "raw_feature_bc_matrix"
                if not (raw_sample_dir / "matrix.mtx.gz").exists() and not (raw_sample_dir / "matrix.mtx").exists():
                    print(f"      -> raw matrix not found at {raw_sample_dir}, skipping this sample")
                    continue

                try:
                    # SoupX pipeline: see rpy2 idiom in SPEC "R Bridge" section.
                    # The core idea: pass filtered counts, raw counts, and cluster labels to R;
                    # SoupX estimates ambient profile and returns corrected counts.
                    #
                    # NOTE: Full SoupX execution requires significant per-sample context
                    # (cluster labels from a quick clustering run, variable gene selection,
                    # etc.). This cell provides the structure and guard logic.
                    # When R is available, PI uncomments the full SoupX block below.
                    print(f"      -> SoupX execution NOT YET IMPLEMENTED in this PR -- structural placeholder; full SoupX (rpy2 %%R + anndata2ri + adjustCounts) deferred to a later PR once R env is set up (see SPEC R Bridge).")
                    adata.obs.loc[sample_mask, "ambient_correction_applied"] = False  # placeholder
                except Exception as e:
                    print(f"      -> SoupX error on {sample_id}: {e}")

print("\nAmbient correction applied:", adata.obs['ambient_correction_applied'].sum(), "cells")

## Filter

Apply basic QC filter thresholds (`min_genes`, `max_genes`, `max_pct_mt`).
These are PI-tunable knobs set in the PARAMS cell at the top.
Doublet removal (via `predicted_doublet`) is **not** part of the basic filter —
doublet labels are recorded in `obs` for downstream consumers to decide.

In [ ]:
# 过滤：应用 PARAMS 中设定的阈值。
print("===== QC 过滤 =====")
cells_before = adata.n_obs
print(f"过滤前细胞数: {cells_before:,}")

# 记录各指标的通过/失败供透明审计。
passed_genes = (adata.obs['n_genes'] >= MIN_GENES) & (adata.obs['n_genes'] <= MAX_GENES)
passed_mt = adata.obs['pct_counts_mt'] <= MAX_PCT_MT
print(f"  min_genes >= {MIN_GENES}:      {(adata.obs['n_genes'] < MIN_GENES).sum():,} 失败")
print(f"  max_genes <= {MAX_GENES}:      {(adata.obs['n_genes'] > MAX_GENES).sum():,} 失败")
print(f"  总合并过滤:                    {(~passed_genes).sum():,} 失败")
print(f"  pct_mt <= {MAX_PCT_MT}:        {(~passed_mt).sum():,} 失败")

# 使用 scanpy 原生 API 执行过滤。
sc.pp.filter_cells(adata, min_genes=MIN_GENES)
sc.pp.filter_cells(adata, max_genes=MAX_GENES)
adata = adata[adata.obs['pct_counts_mt'] <= MAX_PCT_MT, :].copy()

cells_after = adata.n_obs
print(f"\n过滤后细胞数:  {cells_after:,}")
print(f"去除细胞数:      {cells_before - cells_after:,} ({100*(cells_before-cells_after)/cells_before:.1f}%)")

## QC visualisation post-filter

Same plots as pre-filter for before/after comparison. PI eyeballs whether
the thresholds removed the right tails without over-trimming.

In [ ]:
# Post-filter QC violin plots.
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, metric in enumerate(["n_genes", "total_counts", "pct_counts_mt"]):
    ax = axes[i]
    sc.pl.violin(adata, keys=metric, groupby="source_dataset",
                rotation=45, ax=ax, show=False)
    ax.set_title(f"{metric} (post-filter)")
plt.tight_layout()
fig.savefig("results/figures/stage2_qc_violin_post.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Post-filter QC scatter plots.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sc.pl.scatter(adata, x="total_counts", y="n_genes", color="pct_counts_mt",
             ax=axes[0], show=False)
axes[0].set_title("total_counts vs n_genes, colored by pct_mt (post-filter)")
sc.pl.scatter(adata, x="n_genes", y="pct_counts_mt", color="total_counts",
             ax=axes[1], show=False)
axes[1].set_title("n_genes vs pct_mt, colored by total_counts (post-filter)")
plt.tight_layout()
fig.savefig("results/figures/stage2_qc_scatter_post.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Structured QC records — plain adata.uns writes (SPEC Run Metadata convention).
import datetime

# qc_skipped: per-source, per-step record of what was skipped and why.
qc_skipped = {}
for src in source_datasets:
    mf = manifests.get(src, {})
    pp_done = mf.get("preprocessing_done", [])
    qc_overrides = mf.get("qc_overrides", {})
    skipped_steps = {}

    if "doublet_removal" in pp_done or qc_overrides.get("doublet_removal", {}).get("skip"):
        reason = qc_overrides.get("doublet_removal", {}).get("reason", "Author preprocessing")
        skipped_steps["doublet_removal"] = reason

    if "basic_filter" in pp_done:
        skipped_steps["basic_filter"] = "Author already applied basic filter"

    if "normalization" in pp_done:
        skipped_steps["normalization"] = "Author already normalized"

    if skipped_steps:
        qc_skipped[src] = skipped_steps

adata.uns['qc_skipped'] = qc_skipped

# qc_heterogeneous: True when any source skipped any step.
adata.uns['qc_heterogeneous'] = len(qc_skipped) > 0

# filter_v1: record the filter run for reproducibility.
adata.uns['filter_v1'] = {
    "params":     {"min_genes": MIN_GENES, "max_genes": MAX_GENES, "max_pct_mt": MAX_PCT_MT},
    "cells_in":   cells_before,
    "cells_out":  cells_after,
    "timestamp":  datetime.datetime.now().isoformat(),
}

print("qc_skipped:", qc_skipped)
print("qc_heterogeneous:", adata.uns['qc_heterogeneous'])
print("filter_v1:", adata.uns['filter_v1'])

In [ ]:
# Memory discipline self-check (one assertion before write — SPEC Memory Discipline).
# Guards the highest-impact memory regression: adata.X becoming dense or losing float32.
# If this ever fails, investigate which upstream operation densified or cast the matrix.
import scipy.sparse as sp
import numpy as np
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, (
    f"adata.X invariants violated: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
)
print("Memory self-check passed: X is sparse CSR float32.")

In [ ]:
# Write the stage checkpoint to disk.
# compression="lzf" is Memory Discipline #4 — faster than gzip,
# ~30% smaller than uncompressed, and preserves sparse CSR layout.
adata.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"Wrote {OUTPUT_PATH}")

# Verify the file was written and is readable.
import os
assert os.path.exists(OUTPUT_PATH), f"Output NOT found: {OUTPUT_PATH}"
print(f"Verified: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH):,} bytes)")

In [ ]:
# Free memory across stage boundaries (Memory Discipline #3).
# Without this, the Jupyter kernel keeps the previous stage's AnnData
# alive when the next stage is run in the same kernel session.
del adata
import gc
gc.collect()
print("Memory released.")